# Problem 71: Infinite Loop Detector

**Difficulty:** Hard | **Framework:** LangGraph

**Categories:** error-recovery, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/infinite-loop-detector).

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must detect when it is repeating the same action and break out of the loop
- The maximum iteration limit must be configurable
- When the loop is broken, the agent must return a meaningful message about what it tried
- The detection must work for both exact repeats and semantically similar repeated actions


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
import operator

class State(TypedDict):
    query: str
    result: str
    actions: Annotated[list[str], operator.add]

def analyze(state: State) -> State:
    return {"actions": ["analyze"], "result": "Need more data"}

def fetch_data(state: State) -> State:
    return {"actions": ["fetch_data"], "result": "Data insufficient"}

def should_continue(state: State) -> str:
    # BUG: Always routes back to analyze — creates an infinite loop
    if state["result"] == "Done":
        return "end"
    return "analyze"

graph = StateGraph(State)
graph.add_node("analyze", analyze)
graph.add_node("fetch_data", fetch_data)
graph.add_edge(START, "analyze")
graph.add_edge("analyze", "fetch_data")
graph.add_conditional_edges("fetch_data", should_continue, {"analyze": "analyze", "end": END})

app = graph.compile()

# Test: This will loop forever
result = app.invoke({"query": "Analyze sales data", "result": "", "actions": []})
print(result["result"])

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Track the history of actions taken and compare new actions against recent history
2. Set a hard iteration cap (e.g. 5) and exit the loop when reached
3. Compare the current tool call and arguments to the previous N calls to detect repetition


## 7. Evaluation Criteria

Check your solution against these criteria:

- Infinite loop is detected
- Loop is broken after N iterations
- Exit message is meaningful
- Non-looping flows still work
